# Lab 07-09 · Mutual Inductance and Transformers

Companion to chapter [07-09](../../chapters/07-09-mutual-inductance-and-transformers.md). Handout: [README.md](README.md).

Coupled coils are solved here from their inductance matrix, written for currents entering the dotted terminals, so the chapter's reflected-impedance and ideal-transformer formulas are being checked, not assumed. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import math
import shutil
import subprocess
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)


def ngspice_values(path):
    out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) >= 2:
            try:
                vals[parts[0].strip()] = float(parts[1].split()[0])
            except (ValueError, IndexError):
                pass
    return vals


def solve(Vs, R1, L1, L2, k, ZL, f):
    # currents entering the dots: primary i1 and secondary j2 (the load current is -j2)
    w = 2 * math.pi * f; M = k * math.sqrt(L1 * L2)
    Z = np.array([[R1 + 1j * w * L1, 1j * w * M], [1j * w * M, ZL + 1j * w * L2]])
    i1, j2 = np.linalg.solve(Z, [Vs, 0])
    return i1, -j2

ngspice available: True


## Part 1 — Coupled coils at any k

Example 07-09.2: 10 V at 1 kHz through 10 Ω, $L_1$ = 10 mH, $L_2$ = 40 mH, 100 Ω load.

**🔮 Predict:** does the load power rise or fall as $k$ goes from 0.5 to 0.99?

In [2]:
for k in (0.1, 0.3, 0.5, 0.7, 0.9, 0.99):
    i1, i2 = solve(10, 10, 10e-3, 40e-3, k, 100, 1e3)
    zin = 10 / i1 - 10
    print(f"k = {k:4.2f}: load power {0.5 * abs(i2) ** 2 * 100:7.4f} W, input impedance past R1 {zin.real:6.2f} {zin.imag:+7.2f}j ohm")
i1, i2 = solve(10, 10, 10e-3, 40e-3, 0.5, 100, 1e3)
assert abs(0.5 * abs(i2) ** 2 * 100 - 0.1012) < 5e-5
if HAVE_NGSPICE:
    print("ngspice:", ngspice_values("ex2_coupled.cir"))

k = 0.10: load power  0.0027 W, input impedance past R1   0.22  +62.29j ohm
k = 0.30: load power  0.0277 W, input impedance past R1   1.94  +57.95j ohm
k = 0.50: load power  0.1012 W, input impedance past R1   5.40  +49.27j ohm
k = 0.70: load power  0.3043 W, input impedance past R1  10.58  +36.25j ohm
k = 0.90: load power  0.7859 W, input impedance past R1  17.48  +18.89j ohm
k = 0.99: load power  0.9941 W, input impedance past R1  21.15   +9.67j ohm
ngspice: {'Doing analysis at TEMP': 27.0, 'i1m': 0.1937227, 'i1p': -72.6474, 'i2m': 0.04499947, 'i2p': -50.9504, 'pl': 0.1012476, 'pl99': 0.9940607, 'rin99': 21.15358}


## Part 2 — Toward the ideal transformer

**🔮 Predict:** with $k$ = 0.9999 and both inductances multiplied by 10, how close are the voltage ratio $V_2/V_1$ and the current ratio $I_1/I_2$ to $\sqrt{L_2/L_1}$ = 2? Then see what happens with larger inductances but only $k$ = 0.99: the leakage reactance $(1 - k^2)\omega L_2$ grows with $L$.

In [3]:
for scale, k in ((1, 0.5), (1, 0.99), (1, 0.9999), (10, 0.9999), (100, 0.99)):
    i1, i2 = solve(10, 0.0, 10e-3 * scale, 40e-3 * scale, k, 100, 1e3)
    v1, v2 = 10, i2 * 100
    print(f"L x{scale:3d}, k = {k}: V2/V1 = {abs(v2) / v1:.4f}, I1/I2 = {abs(i1) / abs(i2):.4f}")
i1, i2 = solve(10, 0.0, 0.1, 0.4, 0.9999, 100, 1e3)
assert abs(abs(i2) * 100 / 10 - 2) < 0.002 and abs(abs(i1) / abs(i2) - 2) < 0.005

L x  1, k = 0.5: V2/V1 = 0.4686, I1/I2 = 4.3050
L x  1, k = 0.99: V2/V1 = 1.9775, I1/I2 = 2.1742
L x  1, k = 0.9999: V2/V1 = 1.9998, I1/I2 = 2.1527
L x 10, k = 0.9999: V2/V1 = 1.9998, I1/I2 = 2.0018
L x100, k = 0.99: V2/V1 = 0.3882, I1/I2 = 2.0202


## Part 3 — Efficiency against load

**🔮 Predict:** for Example 07-09.4's transformer (core 90 W, full-load copper 497 W), at what load is the efficiency highest?

In [4]:
S = 50e3
x = np.linspace(0.01, 1.2, 2000)
for core, cu, name in ((90, 497, "G"), (30, 560, "A")):
    eta = x * S / (x * S + core + x**2 * cu)
    i = np.argmax(eta)
    print(f"design {name}: maximum efficiency {eta[i] * 100:.3f}% at {x[i] * 100:.1f}% load (formula {math.sqrt(core / cu) * 100:.1f}%)")
assert abs(math.sqrt(90 / 497) - 0.4255) < 1e-4

design G: maximum efficiency 99.161% at 42.6% load (formula 42.6%)
design A: maximum efficiency 99.484% at 23.1% load (formula 23.1%)


## Part 4 — A wireless link

Problem 07-09.12: 24 µH coils with 0.2 Ω each at 360 kHz, a 5 Ω load, the secondary tuned by a series capacitor.

**🔮 Predict:** how does the efficiency fall as $k$ falls from 0.3 to 0.01? What does removing the tuning capacitor do?

In [5]:
L, f, R = 24e-6, 360e3, 0.2
w = 2 * math.pi * f; Ct = 1 / (w * w * L)


def link(k, tuned=True):
    M = k * L
    Z2 = R + 5 + (1 / (1j * w * Ct) if tuned else 0) + 1j * w * L
    j2 = -1j * w * M / Z2                      # 1 A into the primary
    Vp = (R + 1j * w * L) + 1j * w * M * j2
    return 0.5 * abs(j2) ** 2 * 5 / (0.5 * Vp.real)


for k in (0.3, 0.25, 0.1, 0.05, 0.03, 0.01):
    print(f"k = {k:4.2f}: tuned {link(k) * 100:5.1f}%, untuned {link(k, False) * 100:5.1f}%")
assert abs(link(0.25) * 100 - 95.6) < 0.05
if HAVE_NGSPICE:
    print("ngspice:", ngspice_values("p12_wireless.cir"))

k = 0.30: tuned  95.8%, untuned  67.2%
k = 0.25: tuned  95.6%, untuned  59.3%
k = 0.10: tuned  92.9%, untuned  19.7%
k = 0.05: tuned  84.3%, untuned   5.8%
k = 0.03: tuned  69.1%, untuned   2.2%
k = 0.01: tuned  21.2%, untuned   0.2%
ngspice: {'Doing analysis at TEMP': 27.0, 'eta25': 95.61398, 'eta10': 92.87628, 'eta03': 69.07077, 'etaun': 59.31648}


## Your turn

1. Two coils measure 60 mH in series-aiding and 20 mH in series-opposing, and one coil alone is 15 mH. Find $M$, the other coil's inductance and $k$.
2. A 1:3 step-up transformer drives 900 Ω. What resistance does the source see?
3. A transformer has 50 W core loss and 800 W full-load copper loss. At what load is it most efficient?

<details><summary>Answers</summary>

1. $4M = 60 - 20$, so $M$ = **10 mH**; $L_1 + L_2 = 40$ mH, so $L_2$ = **25 mH**; $k = 10/\sqrt{15\times25}$ = **0.516**.
2. $(1/3)^2\times900$ = **100 Ω**.
3. $\sqrt{50/800}$ = **25%** of rated load.

</details>

In [6]:
M = (60 - 20) / 4; L2 = 40 - 15
print(f"1) M = {M} mH, L2 = {L2} mH, k = {M / math.sqrt(15 * L2):.3f}")
print(f"2) {900 / 9:.0f} ohm")
print(f"3) {math.sqrt(50 / 800) * 100:.0f}% load")
assert abs(M / math.sqrt(15 * L2) - 0.516) < 5e-4

1) M = 10.0 mH, L2 = 25 mH, k = 0.516
2) 100 ohm
3) 25% load
